# FlexBend identification comparison

Open-loop test rollouts of N4SID, Koopman with linear $C$, and Deep Koopman (nonlinear decoder) on held-out pch7–pch9.


In [5]:
import os
# OpenMP/MKL + PyTorch on macOS segfaults the Jupyter kernel during backward
# unless thread counts are pinned before numpy/torch are imported.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, os.path.join(os.getcwd()))
sys.path.insert(0, os.path.join(project_root, "src"))
IDENT_DIR = Path.cwd()
DATA_DIR = IDENT_DIR.parent / "data"

import helper
from data_utils import NSTEPS, TS, load_flexy_splits, save_flexy_npz
from helper.koopman import (
    LossWeights,
    TrainConfig,
    build_koopman_problem,
    evaluate_mae_physical,
    extract_matrices,
    get_data_loaders,
    train_koopman,
)

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from helper.koopman import build_koopman_problem, rollout_predictions

experiments, train_exps, dev_exps, test_exps, train, dev, test = load_flexy_splits()
scaler = joblib.load(DATA_DIR / "scaler_flexy.pkl")
scalerU = joblib.load(DATA_DIR / "scalerU_flexy.pkl")
ny, nu = 1, 1


def ss_lsim(A, B, C, D, U, x0):
    N = U.shape[1]
    y = np.zeros((C.shape[0], N))
    x = np.asarray(x0, dtype=float).reshape(-1)
    for k in range(N):
        uk = U[:, k]
        y[:, k] = C @ x + D @ uk
        x = A @ x + B @ uk
    return y


def load_deep_koopman():
    state = torch.load(DATA_DIR / "model_flexy_C_False.pth", map_location="cpu", weights_only=False)
    nz = int(state["nodes.3.nodes.0.callable.K.weight"].shape[0])
    problem, _, _, _ = build_koopman_problem(
        ny=ny, nu=nu, nz=nz, matrix_C=False,
        encoder_depth=2, width_mult=1.0, nonlin="elu",
        loss_weights=LossWeights(), nsteps=4,
    )
    problem.load_state_dict(state)
    problem.eval()
    return problem


def rollout_linear(A, B, C, D, Y, U):
    y_s = scaler.transform(Y)
    u_s = scalerU.transform(U)
    x0 = np.linalg.pinv(C) @ y_s[:1].T
    yhat_s = ss_lsim(A, B, C, D, u_s.T, x0)
    return scaler.inverse_transform(yhat_s.T)


def rollout_dk(problem, Y, U):
    y_s = torch.tensor(scaler.transform(Y)[None], dtype=torch.float32)
    u_s = torch.tensor(scalerU.transform(U)[None], dtype=torch.float32)
    data = {"Y": y_s, "Y0" : y_s[:, 0:1, :], "U": u_s}
    true_s, pred_s = rollout_predictions(problem, data)
    n = pred_s.shape[0]
    return scaler.inverse_transform(true_s), scaler.inverse_transform(pred_s), n


A_c = np.load(DATA_DIR / "A_flexy_C_True.npy")
B_c = np.load(DATA_DIR / "B_flexy_C_True.npy")
C_c = np.load(DATA_DIR / "C_flexy_C_True.npy")
D_c = np.zeros((1, 1))
A_n = np.load(DATA_DIR / "A_flexy_n4sid.npy")
B_n = np.load(DATA_DIR / "B_flexy_n4sid.npy")
C_n = np.load(DATA_DIR / "C_flexy_n4sid.npy")
D_n = np.load(DATA_DIR / "D_flexy_n4sid.npy")
problem_dk = load_deep_koopman()

rows = []
fig, axs = plt.subplots(len(test_exps), 1, figsize=(10, 3.2 * len(test_exps)), sharex=False)
if len(test_exps) == 1:
    axs = [axs]
for ax, exp in zip(axs, test_exps):
    Y, U = exp["Y"], exp["U"]
    plant_dk, pred_dk, n_dk = rollout_dk(problem_dk, Y, U)
    pred_c = rollout_linear(A_c, B_c, C_c, D_c, Y, U)
    pred_n = rollout_linear(A_n, B_n, C_n, D_n, Y, U)
    n = min(n_dk, pred_c.shape[0] - 1, pred_n.shape[0] - 1, Y.shape[0] - 1)
    plant = Y[1 : 1 + n, 0]
    pred_dk = pred_dk[:n, 0]
    pred_c = pred_c[1 : 1 + n, 0]
    pred_n = pred_n[1 : 1 + n, 0]
    t = np.arange(n) * TS
    ax.plot(t, plant, color="k", lw=1.2, label="plant")
    ax.plot(t, pred_dk, "-.", color="#8B0000", lw=1.6, label="Deep Koopman")
    ax.plot(t, pred_c, "--", color="#FFA500", lw=1.2, label="Koopman linear C")
    ax.plot(t, pred_n, ":", color="#008000", lw=1.2, label="N4SID")
    ax.set_ylabel("y")
    ax.set_title(f"pch{exp['id']}: {exp['label']}")
    ax.grid(True)
    mae = {
        "Deep Koopman": float(np.mean(np.abs(plant - pred_dk))),
        "Koopman linear C": float(np.mean(np.abs(plant - pred_c))),
        "N4SID": float(np.mean(np.abs(plant - pred_n))),
    }
    rows.append((exp["id"], mae))
    print(f"pch{exp['id']} MAE", mae)
axs[0].legend(loc="best")
axs[-1].set_xlabel("t [s]")
fig.tight_layout()
fig.savefig(DATA_DIR / "ident_test_comparison.png", dpi=150)

print()
print(f"{'pch':<8} {'Deep Koopman':>14} {'linear C':>14} {'N4SID':>14}")
mean = {k: 0.0 for k in rows[0][1]}
for exp_id, mae in rows:
    print(f"{exp_id:<8} {mae['Deep Koopman']:14.4f} {mae['Koopman linear C']:14.4f} {mae['N4SID']:14.4f}")
    for k, v in mae.items():
        mean[k] += v
n_e = len(rows)
print(f"{'mean':<8} {mean['Deep Koopman']/n_e:14.4f} {mean['Koopman linear C']/n_e:14.4f} {mean['N4SID']/n_e:14.4f}")


RuntimeError: Error(s) in loading state_dict for Problem:
	Unexpected key(s) in state_dict: "nodes.0.callable.linear.3.weight", "nodes.0.callable.linear.3.bias", "nodes.1.callable.linear.3.weight", "nodes.1.callable.linear.3.bias", "nodes.4.callable.linear.3.weight", "nodes.4.callable.linear.3.bias". 
	size mismatch for nodes.0.callable.linear.1.weight: copying a param with shape torch.Size([3, 2]) from checkpoint, the shape in current model is torch.Size([4, 2]).
	size mismatch for nodes.0.callable.linear.1.bias: copying a param with shape torch.Size([3]) from checkpoint, the shape in current model is torch.Size([4]).
	size mismatch for nodes.0.callable.linear.2.weight: copying a param with shape torch.Size([4, 3]) from checkpoint, the shape in current model is torch.Size([7, 4]).
	size mismatch for nodes.0.callable.linear.2.bias: copying a param with shape torch.Size([4]) from checkpoint, the shape in current model is torch.Size([7]).
	size mismatch for nodes.1.callable.linear.1.weig